# Few-shot classification
This notebook demonstrates how to use the few-shot classification baselines.

There are two baselines:
1. Centroid-based classifier
2. Nearest neighbour classifier

The centroid-based classifier computes the class prototype by averaging the features of the support set. The class prototype is then used to classify the query set.

The nearest neighbour classifier classifies the query set by finding the nearest neighbour in the support set.

The notebook consists of the following steps:
1. Precompute features
2. Evaluate the baselines

The few-shot baselines require pre-extracted features. The features can be extracted using the feature extraction script.

### Setup

In [ ]:
from scripts.baselines.few_shot.feature_generation import generate_embeddings
import importlib
from types import SimpleNamespace
from typing import Sequence
import argparse
import os
from pathlib import Path

import numpy as np
import torch
import yaml
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel
from torchvision import transforms as tfms
from timm.data import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD
import pandas as pd
from PIL import Image
import open_clip

from pathlib import Path
from typing import Tuple, Any, Dict, List, Optional, Union
from types import SimpleNamespace

import torch
import torchvision.transforms as T
import matplotlib.pyplot as plt
import pandas as pd
import os
import yaml

from fgvc.datasets import ImageDataset



DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ['clip', 'dinov2', 'bioclip']
model_name = 'bioclip'
# ['centroid', 'nn']
classifier_name = 'centroid'
split = 'test'

# path to fungitatsic dataset
data_path = ''
# path to store precomputed features
feature_path = ''
# path to saved results
path_out = "../../../out/"

## 1. Precompute features

### 1.1 Define feature extractors

In [ ]:
class FeatureExtractor(torch.nn.Module):
    def __init__(self, device):
        super(FeatureExtractor, self).__init__()
        self.device = device

    def extract_features(self, image_path):
        raise NotImplementedError

    def load(self):
        raise NotImplementedError

    @staticmethod
    def normalize_embedding(embs):
        """
        Normalize the embedding to -1, 1 range
        :return:
        """
        embs = embs.float()
        norm_features = torch.nn.functional.normalize(embs, dim=1, p=2)
        return norm_features

    @staticmethod
    def quantize_normalized_embedding(embs):
        """
        Quantize the normalized embedding to 8 bit unsigned integers
        :return:
        """
        embs = embs.float()

        assert embs.min() >= -1 and embs.max() <= 1, 'Embeddings must be normalized to -1, 1 range'

        # quantize the -1, 1 range to 8 bit u-integers
        image_features = ((embs + 1) * 127.5).to(torch.uint8).detach().cpu().numpy()
        return image_features


class DinoV2(FeatureExtractor):
    def __init__(self, device):
        super(DinoV2, self).__init__(device)
        self.model = None
        self.transform = self.get_transform()

    def load(self, model_name='vitb14_reg'):
        if model_name == 'vitb14_reg':
            model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14_reg')
        else:
            raise ValueError(f'Unknown model kind: {model_name}')

        model.eval()
        model.to(self.device)

        self.model = model

    def extract_features(self, image):
        """

        :param model:
        :param image_tensor:
        :return:
        """

        if self.model is None:
            raise ValueError('Model not loaded')

        # get the features
        image_tensor = self.transform(image).unsqueeze(0)
        features = self.model(image_tensor.to(self.device))
        norm_features = self.normalize_embedding(features)
        return norm_features

    @staticmethod
    def get_transform(resize_size: int = 224,
                mean: Sequence[float] = IMAGENET_DEFAULT_MEAN,
                std: Sequence[float] = IMAGENET_DEFAULT_STD,
        ):
        transforms_list = [
            tfms.Resize((resize_size, resize_size), interpolation=tfms.InterpolationMode.BICUBIC),
            tfms.ToTensor(),
            tfms.Normalize(mean=mean, std=std)
        ]
        return tfms.Compose(transforms_list)


class CLIP(FeatureExtractor):
    def __init__(self, device):
        super(CLIP, self).__init__(device)
        self.model = None
        self.processor = None
        # pil image resize to 224, 224
        self.size = 224, 224

    def load(self, model_name='clip-vit-base-patch32'):
        # 'clip-vit-base-patch32'
        # clip-vit-large-patch14
        model = CLIPModel.from_pretrained(f"openai/{model_name}")
        processor = CLIPProcessor.from_pretrained(f"openai/{model_name}")

        model.to(self.device)

        self.model = model
        self.processor = processor

    def extract_features(self, image):
        if self.model is None:
            raise ValueError('Model not loaded')

        image = image.resize(self.size, Image.BICUBIC)
        # TODO put image on gpu before processing (check what normalization is expected for tensors)
        image_tensor_proc = self.processor(images=image, return_tensors='pt').pixel_values

        # get the features
        features = self.model.get_image_features(pixel_values=image_tensor_proc.to(self.device))
        norm_features = self.normalize_embedding(features)
        return norm_features


class BioCLIP(CLIP):
    def load(self, model_name='bioclip'):
        # bioclip-vit-b-16-inat-only
        model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms('hf-hub:imageomics/bioclip')
        # tokenizer = open_clip.get_tokenizer('hf-hub:imageomics/bioclip')
        self.processor = preprocess_val
        model.to(self.device)
        self.model = model

    def extract_features(self, image):
        if self.model is None:
            raise ValueError('Model not loaded')

        image = image.resize(self.size, Image.BICUBIC)
        image_tensor_proc = self.processor(image)[None]

        # get the features
        features = self.model.encode_image(image_tensor_proc.to(self.device))
        norm_features = self.normalize_embedding(features)
        return norm_features


def get_model(model_name):
    if model_name == 'clip':
        model = CLIP(device=DEVICE)
    elif model_name == 'dinov2':
        model = DinoV2(device=DEVICE)
    elif model_name == 'bioclip':
        model = BioCLIP(device=DEVICE)
    else:
        raise ValueError(f'Unknown model kind: {model_name}')

    model.load()
    model.eval()
    return model

### 1.2 Define dataset class for FungiTastic

In [ ]:
class FungiTastic(ImageDataset):
    """
    Dataset class for the Danish Fungi dataset.

    Dataframe keys: ['eventDate', 'year', 'month', 'day', 'habitat', 'countryCode',
                     'scientificName', 'kingdom', 'phylum', 'class', 'order', 'family',
                     'genus', 'specificEpithet', 'hasCoordinate', 'species',
                     'iucnRedListCategory', 'substrate', 'latitude', 'longitude',
                     'coorUncert', 'observationID', 'region', 'district', 'filename',
                     'category_id', 'metaSubstrate', 'poisonous', 'elevation', 'landcover',
                     'biogeographicalRegion', 'image_path']
    """
    SUBSET2SIZES: Dict[str, List[str]] = {
        'all': ['300', '500'],
        'FewShot': ['300', '500'],
        'Mini': ['300', '500', '720', 'fullsize'],
    }

    SUBSET2TASKS: Dict[str, List[str]] = {
        'all': ['open', 'closed'],
        'FewShot': ['closed'],
        'Mini': ['open', 'closed'],
    }

    SUBSET2SPLITS: Dict[str, List[str]] = {
        'all': ['train', 'val', 'test', 'dna'],
        'FewShot': ['train', 'val', 'test'],
        'Mini': ['train', 'val', 'test', 'dna'],
    }

    SPLIT2STR: Dict[str, str] = {
        'train': 'Train',
        'val': 'Val',
        'test': 'Test',
        'dna': 'DNA-Test'
    }

    TASK2STR: Dict[str, str] = {
        'open': 'OpenSet',
        'closed': 'ClosedSet',
    }

    SUBSETS = SUBSET2SIZES.keys()

    def __init__(self, root: str, data_subset: str = 'Mini', split: str = 'val', size: str = '300',
                 task: str = 'closed', transform: T.Compose = None, **kwargs):
        df = self.get_df(
            data_path=root,
            split=split,
            size=size,
            task=task,
            data_subset=data_subset
        )

        assert "image_path" in df
        self.df = df
        self.transform = transform
        self.data_subset = data_subset
        self.split = split
        self.task = task

        if split in ['train', 'val']:
            assert "category_id" in df
            category_id2label = df.groupby('category_id')['species'].unique().to_dict()
            unknown_species = list(category_id2label.get(-1, []))
            self.unkwnown_id = -1
            self.category_id2label = {k: v[0] for k, v in category_id2label.items()}
            self.label2category_id = {v: k for k, v in self.category_id2label.items()}
            category_id2label[self.unkwnown_id] = unknown_species
            for unk_spec in unknown_species:
                self.label2category_id[unk_spec] = self.unkwnown_id

            self.n_classes = len(self.df['category_id'].unique())

    def get_class_id(self, idx: int) -> int:
        """
        Get class id of i-th element in the dataset.

        Args:
            idx (int): Index of the element.

        Returns:
            int: Class ID of the element.
        """
        return self.df["category_id"].iloc[idx]

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, Optional[int], str]:
        """
        Get item (image, class ID, and file path) by index.

        Args:
            idx (int): Index of the element.

        Returns:
            Tuple[torch.Tensor, Optional[int], str]: Image tensor, class ID, and file path.
        """
        if self.split in ['train', 'val']:
            return super().__getitem__(idx)
        else:
            image, file_path = self.get_image(idx)
            image = self.apply_transforms(image)
            return image, None, file_path

    @staticmethod
    def check_params(data_subset: str, split: str, size: str, task: str) -> None:
        """
        Check the validity of dataset parameters.

        Args:
            data_subset (str): Subset of the dataset.
            split (str): Data split.
            size (str): Image size.
            task (str): Task type.

        Raises:
            AssertionError: If any parameter is invalid.
        """
        assert data_subset in FungiTastic.SUBSETS, f"Invalid subset: {data_subset}. Available subsets are: {FungiTastic.SUBSETS}"
        assert split in ['train', 'val', 'test',
                         'dna'], f"Invalid split: {split}. Available splits are: ['train', 'val', 'test', 'dna']"
        assert size in FungiTastic.SUBSET2SIZES[data_subset], (f"Invalid size: {size}. Available sizes for subset "
                                                               f"{data_subset} are: {FungiTastic.SUBSET2SIZES[data_subset]}")
        assert task in FungiTastic.SUBSET2TASKS[data_subset], (f"Invalid task: {task}. Available tasks for subset "
                                                               f"{data_subset} are: {FungiTastic.SUBSET2TASKS[data_subset]}")
        assert split in FungiTastic.SUBSET2SPLITS[data_subset], (f"Invalid split: {split}. Available splits for subset "
                                                                 f"{data_subset} are: {FungiTastic.SUBSET2SPLITS[data_subset]}")
        assert not (task == 'open' and split == 'dna'), "Open set task is not available for DNA split"

    @staticmethod
    def get_df(data_path: str, split: str = 'val', task: str = 'closed', size: str = '300',
               data_subset: str = 'Mini') -> pd.DataFrame:
        """
        Get the dataframe for the specified dataset parameters.

        Args:
            data_path (str): Path to the dataset.
            split (str): Data split.
            task (str): Task type.
            size (str): Image size.
            data_subset (str): Subset of the dataset.

        Returns:
            pd.DataFrame: Dataframe containing the dataset metadata.
        """
        FungiTastic.check_params(data_subset, split, size, task)

        subfolder_str = f'FungiTastic-{data_subset}' if data_subset != 'all' else 'FungiTastic'
        data_subset_str = f'-{data_subset}' if data_subset != 'all' else ''
        task_str = f'-{FungiTastic.TASK2STR[task]}' if (data_subset != 'FewShot' and split != 'train') else ''

        df_path = os.path.join(
            data_path,
            "metadata",
            subfolder_str,
            f"FungiTastic{data_subset_str}{task_str}-{FungiTastic.SPLIT2STR[split]}.csv",
        )
        df = pd.read_csv(df_path)
        df["image_path"] = df.filename.apply(
            lambda x: os.path.join(data_path, subfolder_str, split, f'{size}p', x)
        )
        return df

    def show_sample(self, idx: int) -> None:
        """
        Display a sample image with its class name and ID.

        Args:
            idx (int): Index of the sample to display.
        """
        image, category_id, file_path = self.__getitem__(idx)
        class_name = self.category_id2label[category_id] if category_id is not None else '[TEST]'
        plt.imshow(image)
        plt.title(f"Class: {class_name}; id: {idx}")
        plt.axis('off')
        plt.show()

    def get_category_idxs(self, category_id: int) -> List[int]:
        """
        Get all indexes of a specific class ID.

        Args:
            category_id (int): Class ID to search for.

        Returns:
            List[int]: List of indexes that belong to the specified class ID.
        """
        return self.df[self.df.category_id == category_id].index.tolist()

### 1.3 Feature generation helper functions

In [ ]:
def generate_embeddings(data_path, feature_path, model_name='clip', data_split='val'):
    model = get_model(model_name)

    splits = [data_split] if data_split != 'all' else ['val', 'test', 'train']

    for split in splits:

        dataset = FungiTastic(
            root=data_path,
            split=split,
            size='300',
            task='closed',
            data_subset='FewShot',
            transform=None,
            )

        if model_name == 'dinov2':
            feature_folder = os.path.join(feature_path, f'{model_name}_vit_b')
        else:
            feature_folder = os.path.join(feature_path, model_name)

        # TODO: run in batch mode https://github.com/openai/CLIP/issues/175
        save_freq = -1

        # if it doesn't exist, create the feature directory
        Path(feature_folder).mkdir(parents=True, exist_ok=True)

        feature_file_full = os.path.join(feature_folder, f'224x224_{split}.h5')
        cols = ['im_name', 'embedding']
        df = pd.DataFrame(columns=cols)

        # idxs = np.arange(int(0.4 * len(nico)), len(nico))[::-1]
        idxs = np.arange(len(dataset))
        im_names, embs = [], []
        for idx in tqdm(idxs):
            im, label, file_path = dataset[idx]

            with torch.no_grad():
                feat = model.extract_features(im)
            feat_quant = model.quantize_normalized_embedding(feat)

            im_names.append(os.path.basename(file_path))
            embs.append(feat_quant)

            if idx % save_freq == 0 and save_freq > 0:
                # concat new dataset to the existing dataframe
                new = pd.DataFrame({'im_name': im_names, 'embedding': embs})
                df = pd.concat([df, new], ignore_index=True)

                # save
                df.to_hdf(feature_file_full, key='df', mode='w')

                # clear the lists
                im_names, embs = [], []

        new = pd.DataFrame({'im_name': im_names, 'embedding': embs})
        df = pd.concat([df, new], ignore_index=True)
        df.to_hdf(feature_file_full, key='df', mode='w')

### 1.3 Precompute features

In [ ]:
# precompute features for the fungi dataset
generate_embeddings(data_path=data_path, model_name=model_name, data_split='train', feature_path=feature_path)
# generate_embeddings(data_path=cfg.data_path, model_name=model_name, data_split='val', feature_path=cfg.feature_path) # not needed
generate_embeddings(data_path=data_path, model_name=model_name, data_split='test', feature_path=feature_path)

## 2. Run the few-shot classifier on the precomputed features
### 2.1 Define the few-shot classifier

In [ ]:
from pathlib import Path

import torch
import pandas as pd
import json
import numpy as np

from fgvc.core.metrics import classification_scores
from tqdm import tqdm
import faiss


"""
Classes only used for evaluation of the models.

"""


class Classifier(torch.nn.Module):
    def __init__(self, device):
        super().__init__()
        self.device = device

        # add option to save the results for further processing
        self.test_results = {}

    def make_prediction(self, x):
        raise NotImplementedError

    @property
    def name(self):
        raise NotImplementedError

    def evaluate(self, dataloader, fast_dev_run=False):
        self.eval()

        with torch.no_grad():
            im_names, predictions, gts, confidences, is_correct, probabilities = [], [], [], [], [], []
            for batch_idx, (embeddings, labels, file_paths) in tqdm(enumerate(dataloader)):
                embeddings = embeddings.to(self.device)
                labels = labels.to(self.device)
                # set ims and labels to device
                if fast_dev_run and fast_dev_run == batch_idx:
                    break
                im_names.extend([Path(fp).name for fp in file_paths])
                gts.extend(labels.cpu().numpy())
                cls, conf, probs = self.make_prediction(embeddings, ret_probs=True)
                probabilities.extend(probs.cpu().numpy() if probs is not None else [])
                predictions.extend(cls.cpu().numpy())
                confidences.extend(conf.cpu().numpy())
                is_correct.extend((cls == labels).cpu().numpy())

        cls_scores = classification_scores(np.array(predictions) if len(probabilities) == 0 else np.array(probabilities),
                                           np.array(gts),
                                           top_k=3 if probs is not None else 1)

        print(cls_scores)

        self.test_results = {
            'im_name': im_names,
            'gt': gts,
            'pred': predictions,
            'conf': confidences,
            'is_correct': is_correct,
            'top1_acc': cls_scores['Accuracy'],
            'f1': cls_scores['F1'],
        }

        if probs is not None:
            self.test_results['top3_acc'] = cls_scores['Recall@3']
        self.test_metrics = cls_scores

    def save_results(self, out_dir, file_name=None):
        # create out dir if it doesn't exist
        out_dir.mkdir(parents=True, exist_ok=True)

        #  save test_results as dataframe
        df = pd.DataFrame(self.test_results)
        out_file = Path(out_dir) / f'{file_name}.csv'
        df.to_csv(out_file, index=False)

        # also save the test_metrics as a separate file
        out_file = Path(out_dir) / f'{file_name}_metrics.json'
        with open(out_file, 'w') as f:
            json.dump(self.test_metrics, f)


class PrototypeClassifier(Classifier):
    def __init__(self, train_embeddings, device='cuda'):
        """
        :param cfg: OmegaConf config object
        :param train_embeddings: list of C torch arrays of shape [N_C, D] where N_C is the number of training samples
        of class C and D is the dimensionality of the embeddings
        """
        super().__init__(device=device)

        # C x D array of class prototypes, make them a parameter so that they are moved to the device
        self.class_prototypes = self.get_prototypes(train_embeddings, mode='centroid')
        self.class_prototypes = torch.nn.Parameter(self.class_prototypes, requires_grad=False)

    def get_prototypes(self, embeddings, mode='centroid'):
        if mode == 'centroid':
            class_prototypes = torch.stack([class_embs.mean(dim=0) for class_embs in embeddings])
        else:
            raise ValueError(f"Unknown prototype classifier mode: {mode}")
        return class_prototypes

    def make_prediction(self, embeddings, plot_sim_hist=False, ret_probs=False):
        # compute the cosine similarity of each embedding to each prototype
        similarities = torch.nn.functional.cosine_similarity(embeddings.unsqueeze(1), self.class_prototypes.unsqueeze(0), dim=-1)
        # get the class with the highest similarity
        cls = torch.argmax(similarities, dim=1)
        probs = torch.nn.functional.softmax(similarities, dim=1)
        # get the confidence of the prediction from softmax
        conf = probs.max(dim=1).values

        if plot_sim_hist:
            import matplotlib.pyplot as plt
            plt.hist(similarities[1].cpu().numpy(), bins=100)
            plt.show()
        if ret_probs:
            return cls, conf, probs
        else:
            return cls, conf


class NNClassifier(Classifier):
    def __init__(self, train_embeddings, device='cuda'):
        """
        :param cfg: config object, namespace
        :param train_embeddings: list of C torch arrays of shape [N_C, D] where N_C is the number of training samples
        of class C and D is the dimensionality of the embeddings
        """
        super().__init__(device=device)

        self.index, self.idx2cls = self.build_index(train_embeddings)

    def make_prediction(self, embeddings, plot_sim_hist=False, ret_probs=False):
        """
        :param embeddings: torch.Tensor of shape (batch_size, n_channels, height, width)
        :return: probabilities of shape (batch_size, n_classes) computed based on
        the similarity of the embeddings to the class prototypes
        """
        # compute the similarity of each embedding to each prototype
        # embeddings - [N, D], class_prototypes - [C, D]
        similarities, indices = self.index.search(embeddings.cpu().numpy(),1)
        # get the classes for the indices
        cls = self.idx2cls[indices.squeeze()]
        # get the confidence of the prediction
        conf = similarities
        probs = None
        return cls, conf, probs

    def build_index(self, train_embeddings):
        idx2cls = np.hstack([np.ones(len(embs)) * i for i, embs in enumerate(train_embeddings)])
        # concatenate the embeddings
        embs = torch.cat(train_embeddings)
        # build the index for cosine similarity search
        index = faiss.IndexFlatIP(embs.shape[1])
        index.add(embs.cpu().numpy())
        return index, idx2cls

    def evaluate(self, dataloader, fast_dev_run=False):
        self.eval()

        with torch.no_grad():
            im_names, predictions, gts, confidences, is_correct, probabilities = [], [], [], [], [], []
            for batch_idx, (embeddings, labels, file_paths) in tqdm(enumerate(dataloader)):
                # set ims and labels to device
                if fast_dev_run and fast_dev_run == batch_idx:
                    break
                im_names.extend([Path(fp).name for fp in file_paths])
                gts.extend(labels)
                cls, conf, probs = self.make_prediction(embeddings, ret_probs=True)
                probabilities.extend(probs if probs is not None else [])
                predictions.extend(cls)
                confidences.extend(conf)
                is_correct.extend((cls == labels.numpy()))

        cls_scores = classification_scores(np.array(predictions) if len(probabilities) == 0 else np.array(probabilities),
                                           np.array(gts),
                                           top_k=3 if probs is not None else 1)

        print(cls_scores)

        self.test_results = {
            'im_name': im_names,
            'gt': gts,
            'pred': predictions,
            'conf': confidences,
            'is_correct': is_correct,
            'top1_acc': cls_scores['Accuracy'],
            'f1': cls_scores['F1'],
        }

        if probs is not None:
            self.test_results['top3_acc'] = cls_scores['Recall@3']
        self.test_metrics = cls_scores

### 2.2 Define helper classes for loading of precomputed features

In [ ]:
class FeatureFungiTasticDataset(FungiTastic):
    def __init__(self, root: str, features_file: str, data_subset: str = 'Mini', split: str = 'val', size: str = '300',
                 task: str = 'closed', transform: T.Compose = None, rescale=True, **kwargs):
        super().__init__(
            root=root,
            data_subset=data_subset,
            split=split,
            size=size,
            task=task,
            transform=transform,
            **kwargs
        )
        embeddings = pd.read_hdf(features_file)

        if rescale:
            embeddings['embedding'] = embeddings['embedding'].apply(self.rescale_embedding)

        self.embeddings = embeddings
        self.emb_dim = self.embeddings['embedding'].iloc[0].shape[1]

    def check_integrity(self):
        #     make sure the embedding im_name is the same as df filename for all samples
        emb_names = self.embeddings['im_name'].values
        df_names = self.df['filename'].values
        assert (emb_names == df_names).all()
        print('Integrity check passed!')

    @staticmethod
    def rescale_embedding(embedding):
        # rescale the embedding from np.uint8 back to the range [-1, 1]
        return embedding / 255.0 * 2 - 1

    def get_embeddings_for_class(self, id):
        # return the embeddings for class class_idx
        class_idxs = self.df[self.df['category_id'] == id].index
        return self.embeddings.iloc[class_idxs]['embedding']

    def __getitem__(self, index: int, ret_image=False) -> Tuple[Any, Any, Any]:
        image, class_id, file_path = super().__getitem__(index)
        emb = torch.tensor(self.embeddings.iloc[index]['embedding'], dtype=torch.float32).squeeze()

        if ret_image:
            return image, emb, class_id, file_path
        else:
            return emb, class_id, file_path

### 2.3 Evaluation code

In [ ]:
def get_dataloader(test_dataset, batch_size=256, num_workers=0):
    test_dataloader = torch.utils.data.DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=False,
    )

    return test_dataloader


def get_classifier_cls(classifier_name):
    if classifier_name == 'centroid':
        return PrototypeClassifier
    elif classifier_name == 'nn':
        return NNClassifier
    else:
        raise ValueError(f"Classifier {classifier_name} not implemented")


def get_classifier_embeddings(dataset_train):
    class_embeddings = []
    empty_classes = []
    n_classes = min(torch.inf, dataset_train.n_classes)
    for cls in range(n_classes):
        cls_embs = dataset_train.get_embeddings_for_class(cls)
        if len(cls_embs) == 0:
            # if no embeddings for class, use zeros
            empty_classes.append(cls)
            class_embeddings.append(torch.zeros(1, dataset_train.emb_dim))
        else:
            class_embeddings.append(torch.tensor(np.vstack(cls_embs.values)))
    return class_embeddings, empty_classes

def test_fungi(path_out, data_path, feature_path, feature_model, classifier_name, split, debug=False):
    features_file_train = os.path.join(feature_path, feature_model, f"224x224_no_micro_train.h5")
    features_file_eval = os.path.join(feature_path, feature_model, f"224x224_no_micro_{split}.h5")

    dataset_train = FeatureFungiTasticDataset(
        root=data_path,
        features_file=features_file_train,
        split='train',
        size='300',
        task='closed',
        data_subset='FewShot',
        transform=None,
    )

    dataset_eval = FeatureFungiTasticDataset(
        root=data_path,
        features_file=features_file_eval,
        split=split,
        size='300',
        task='closed',
        data_subset='FewShot',
        transform=None,
    )

    exp_name = f"eval_{feature_model}_{split}_{classifier_name}"

    print(f"Evaluating {exp_name}")

    dataloader = get_dataloader(test_dataset=dataset_eval)

    class_embeddings, _ = get_classifier_embeddings(dataset_train)

    classifier = get_classifier_cls(classifier_name)(class_embeddings, device='cpu')
    # classifier.cuda()

    #  if True, runs 1 train/val batch only in trainer.fit, n batches if set to n
    fast_dev_run = 3 if debug else False

    result_dir = Path(path_out) / 'results' / 'fs' / split
    classifier.evaluate(dataloader=dataloader, fast_dev_run=fast_dev_run)
    classifier.save_results(out_dir=result_dir, file_name=f'{exp_name}')

### 2.4 Run evaluation

In [ ]:
test_fungi(path_out=path_out, data_path=data_path, feature_path=feature_path, feature_model=model_name,
           classifier_name=classifier_name, split=split, debug=False)

